In [1]:
# ============================================================
# STEAM GAMES ANALYTICS DASHBOARD
# Step 1: Data Cleaning & SQL-Ready Preparation
# ============================================================

import pandas as pd
import numpy as np
import sqlite3
import os

# --- 1.1 Load the Dataset ---
# Update this filename to match your downloaded file
files = [f for f in os.listdir('.') if f.endswith('.csv')]
print(f"CSV files found: {files}")

# Load the first CSV found (update if needed)
df = pd.read_csv(files[0])
print(f"\nDataset loaded! Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# --- 1.2 Explore the Data ---
print("\n" + "=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")

# --- 1.3 Clean the Data ---
# Drop rows where critical fields are missing
print("\n" + "=" * 60)
print("CLEANING DATA...")
print("=" * 60)

# Keep a copy of original shape
original_shape = df.shape[0]

# Drop duplicates
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape[0]} rows (removed {original_shape - df.shape[0]})")

# Show column names so we can decide what to keep
print(f"\nAll columns available:\n{df.columns.tolist()}")
print(f"\nDataset ready for exploration! Shape: {df.shape}")

# --- 1.4 Save Cleaned Data ---
# Save as CSV for Tableau
df.to_csv('steam_games_cleaned.csv', index=False)
print(f"\nSaved cleaned data as 'steam_games_cleaned.csv'")

# --- 1.5 Load into SQLite Database ---
# This lets you practice SQL queries on the data
conn = sqlite3.connect('steam_games.db')
df.to_sql('games', conn, if_exists='replace', index=False)
print("Loaded data into SQLite database 'steam_games.db'")

# --- 1.6 Test SQL Queries ---
print("\n" + "=" * 60)
print("TESTING SQL QUERIES")
print("=" * 60)

# Query 1: Total games count
result = pd.read_sql("SELECT COUNT(*) as total_games FROM games", conn)
print(f"\nTotal games in database: {result['total_games'].values[0]}")

# Query 2: Top 10 most reviewed games
print("\nTop 10 games by review count (if column exists):")
try:
    # Try common column names for reviews/ratings
    cols = df.columns.tolist()
    print(f"Available columns to query: {cols[:10]}...")
except Exception as e:
    print(f"Note: {e}")

conn.close()
print("\n Step 1 complete! Data is cleaned and ready for Tableau.")
print("Next: Open 'steam_games_cleaned.csv' in Tableau to start building your dashboard.")

CSV files found: ['games_march2025_full.csv']

Dataset loaded! Shape: (94948, 47)

Columns: ['appid', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags', 'pct_pos_total', 'num_reviews_total', 'pct_pos_recent', 'num_reviews_recent']

First 3 rows:
    appid                 name release_date  required_age  price  dlc_count  \
0     730     Counter-Strike 2   2012-08-21             0    0.0          

In [2]:
# ============================================================
# Step 2: Explore Columns & Create Tableau-Ready Dataset
# ============================================================

# See ALL column names
print("ALL 47 COLUMNS:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3}. {col}")

# Check key columns
print("\n" + "=" * 60)
print("KEY STATS")
print("=" * 60)
print(f"\nPrice range: ${df['price'].min()} - ${df['price'].max()}")
print(f"Free games: {(df['price'] == 0).sum()} ({(df['price'] == 0).mean()*100:.1f}%)")
print(f"Date range: {df['release_date'].min()} to {df['release_date'].max()}")
print(f"\nSample of unique values in key columns:")

# Check which columns have useful data
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    if pct < 100:
        print(f"  {col}: {pct:.0f}% filled")

ALL 47 COLUMNS:
  1. appid
  2. name
  3. release_date
  4. required_age
  5. price
  6. dlc_count
  7. detailed_description
  8. about_the_game
  9. short_description
 10. reviews
 11. header_image
 12. website
 13. support_url
 14. support_email
 15. windows
 16. mac
 17. linux
 18. metacritic_score
 19. metacritic_url
 20. achievements
 21. recommendations
 22. notes
 23. supported_languages
 24. full_audio_languages
 25. packages
 26. developers
 27. publishers
 28. categories
 29. genres
 30. screenshots
 31. movies
 32. user_score
 33. score_rank
 34. positive
 35. negative
 36. estimated_owners
 37. average_playtime_forever
 38. average_playtime_2weeks
 39. median_playtime_forever
 40. median_playtime_2weeks
 41. discount
 42. peak_ccu
 43. tags
 44. pct_pos_total
 45. num_reviews_total
 46. pct_pos_recent
 47. num_reviews_recent

KEY STATS

Price range: $0.0 - $999.98
Free games: 19420 (20.5%)
Date range: 1997-06-30 to 2025-03-10

Sample of unique values in key columns:
  name:

In [3]:
# ============================================================
# Step 3: Create Tableau-Ready Dataset
# ============================================================

# First, let's see remaining columns (25-47)
print("Columns 25-47:")
for i, col in enumerate(df.columns[24:], 25):
    print(f"{i}. {col}")

# Select the most valuable columns for dashboard
tableau_cols = [
    'name', 'release_date', 'price', 'required_age',
    'dlc_count', 'metacritic_score', 'achievements',
    'recommendations', 'windows', 'mac', 'linux',
    'reviews'
]

# Check if these columns with common names exist
all_cols = df.columns.tolist()
print(f"\nAll columns: {all_cols}")

# Let's also check for genre, category, developer, publisher columns
genre_cols = [c for c in all_cols if 'genre' in c.lower() or 'categor' in c.lower() 
              or 'tag' in c.lower() or 'develop' in c.lower() or 'publish' in c.lower()
              or 'positive' in c.lower() or 'negative' in c.lower() or 'owner' in c.lower()
              or 'player' in c.lower() or 'peak' in c.lower()]
print(f"\nKey business columns found: {genre_cols}")

# Select final columns for Tableau
keep_cols = [c for c in tableau_cols if c in all_cols] + genre_cols
keep_cols = list(set(keep_cols))  # remove duplicates
print(f"\nFinal columns selected ({len(keep_cols)}): {sorted(keep_cols)}")

# Create Tableau dataset
df_tableau = df[keep_cols].copy()

# Parse release_date to extract year
if 'release_date' in df_tableau.columns:
    df_tableau['release_date'] = pd.to_datetime(df_tableau['release_date'], errors='coerce')
    df_tableau['release_year'] = df_tableau['release_date'].dt.year
    df_tableau['release_month'] = df_tableau['release_date'].dt.month

# Create price category
if 'price' in df_tableau.columns:
    df_tableau['price_category'] = pd.cut(df_tableau['price'], 
        bins=[-1, 0, 4.99, 14.99, 29.99, 59.99, float('inf')],
        labels=['Free', 'Under $5', '$5-$15', '$15-$30', '$30-$60', '$60+'])

print(f"\nTableau dataset shape: {df_tableau.shape}")
print(f"\nSample data:")
print(df_tableau.head())

# Save for Tableau
df_tableau.to_csv('steam_tableau_ready.csv', index=False)
print(f"\nSaved as 'steam_tableau_ready.csv' — open this file in Tableau!")

Columns 25-47:
25. packages
26. developers
27. publishers
28. categories
29. genres
30. screenshots
31. movies
32. user_score
33. score_rank
34. positive
35. negative
36. estimated_owners
37. average_playtime_forever
38. average_playtime_2weeks
39. median_playtime_forever
40. median_playtime_2weeks
41. discount
42. peak_ccu
43. tags
44. pct_pos_total
45. num_reviews_total
46. pct_pos_recent
47. num_reviews_recent

All columns: ['appid', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2

FREE vs PAID GAMES:
type  count  avg_positive_reviews  avg_peak_players
Free  19420                1436.0             183.0
Paid  75528                1162.0              70.0


TOP 10 PUBLISHERS:
primary_publisher
[]                     5780
['Big Fish Games']      484
['8floor']              237
['EroticGamesClub']     196
['Conglomerate 5']      187
['HH-Games']            168
['Laush Studio']        160
['Choice of Games']     158
['Kagura Games']        155
['Sekai Project']       152


TOP 10 GAMES BY PEAK CONCURRENT PLAYERS:
                       name  peak_ccu  positive  negative  price primary_genre
           Counter-Strike 2   1212356   7480813   1135108   0.00     ['Action'
       Monster Hunter Wilds    703236     74671     58069  69.99     ['Action'
        PUBG: BATTLEGROUNDS    616738   1487960   1024436   0.00     ['Action'
                     Dota 2    555977   1998462    451338   0.00     ['Action'
              Split Fiction    201694     17127       314  49.99   

In [8]:
# ============================================================
# STEAM GAMES — COMPLETE SQL ANALYSIS (All Queries)
# ============================================================

import sqlite3

# --- Create new columns ---
df_tableau['primary_genre'] = df_tableau['genres'].astype(str).str.split(',').str[0].str.strip()
df_tableau['primary_publisher'] = df_tableau['publishers'].astype(str).str.split(',').str[0].str.strip()
df_tableau['primary_developer'] = df_tableau['developers'].astype(str).str.split(',').str[0].str.strip()

# --- Load into SQLite ---
conn = sqlite3.connect('steam_games.db')
df_tableau.to_sql('steam', conn, if_exists='replace', index=False)

# ============================================================
# QUERY 1: Games Released by Year
# ============================================================
q1 = pd.read_sql("""
    SELECT release_year, COUNT(*) as game_count, 
           ROUND(AVG(price), 2) as avg_price
    FROM steam 
    WHERE release_year BETWEEN 2010 AND 2025
    GROUP BY release_year 
    ORDER BY release_year
""", conn)
print("=" * 60)
print("QUERY 1: GAMES BY YEAR")
print("=" * 60)
print(q1.to_string(index=False))

# ============================================================
# QUERY 2: Top 10 Genres by Game Count
# ============================================================
print("\n" + "=" * 60)
print("QUERY 2: TOP 10 GENRES")
print("=" * 60)
genre_counts = df_tableau['primary_genre'].value_counts().head(10)
print(genre_counts.to_string())

# ============================================================
# QUERY 3: Price Category Distribution
# ============================================================
print("\n" + "=" * 60)
print("QUERY 3: PRICE DISTRIBUTION")
print("=" * 60)
price_dist = df_tableau['price_category'].value_counts().sort_index()
print(price_dist.to_string())

# ============================================================
# QUERY 4: Top 10 Most Reviewed Games
# ============================================================
q4 = pd.read_sql("""
    SELECT name, positive, negative, 
           (positive + negative) as total_reviews,
           ROUND(positive * 100.0 / (positive + negative), 1) as positive_pct,
           price, peak_ccu
    FROM steam 
    WHERE positive > 0 AND negative > 0
    ORDER BY total_reviews DESC 
    LIMIT 10
""", conn)
print("\n" + "=" * 60)
print("QUERY 4: TOP 10 MOST REVIEWED GAMES")
print("=" * 60)
print(q4.to_string(index=False))

# ============================================================
# QUERY 5: Free vs Paid Games Comparison
# ============================================================
q5 = pd.read_sql("""
    SELECT 
        CASE WHEN price = 0 THEN 'Free' ELSE 'Paid' END as type,
        COUNT(*) as count,
        ROUND(AVG(positive), 0) as avg_positive_reviews,
        ROUND(AVG(peak_ccu), 0) as avg_peak_players
    FROM steam
    GROUP BY type
""", conn)
print("\n" + "=" * 60)
print("QUERY 5: FREE vs PAID GAMES")
print("=" * 60)
print(q5.to_string(index=False))

# ============================================================
# QUERY 6: Top 10 Publishers by Game Count
# ============================================================
print("\n" + "=" * 60)
print("QUERY 6: TOP 10 PUBLISHERS")
print("=" * 60)
print(df_tableau['primary_publisher'].value_counts().head(10).to_string())

# ============================================================
# QUERY 7: Top 10 Games by Peak Concurrent Players
# ============================================================
q7 = pd.read_sql("""
    SELECT name, peak_ccu, positive, negative, price, primary_genre
    FROM steam
    WHERE peak_ccu > 0
    ORDER BY peak_ccu DESC
    LIMIT 10
""", conn)
print("\n" + "=" * 60)
print("QUERY 7: TOP 10 GAMES BY PEAK PLAYERS")
print("=" * 60)
print(q7.to_string(index=False))

# ============================================================
# QUERY 8: Average Price by Genre
# ============================================================
q8 = pd.read_sql("""
    SELECT primary_genre, 
           COUNT(*) as game_count,
           ROUND(AVG(price), 2) as avg_price,
           ROUND(AVG(positive), 0) as avg_positive_reviews
    FROM steam
    WHERE primary_genre != 'nan'
    GROUP BY primary_genre
    HAVING game_count > 100
    ORDER BY avg_price DESC
    LIMIT 10
""", conn)
print("\n" + "=" * 60)
print("QUERY 8: AVERAGE PRICE BY GENRE")
print("=" * 60)
print(q8.to_string(index=False))

# ============================================================
# QUERY 9: Platform Support (Windows/Mac/Linux)
# ============================================================
q9 = pd.read_sql("""
    SELECT 
        SUM(CASE WHEN windows = 1 OR windows = 'True' THEN 1 ELSE 0 END) as windows_count,
        SUM(CASE WHEN mac = 1 OR mac = 'True' THEN 1 ELSE 0 END) as mac_count,
        SUM(CASE WHEN linux = 1 OR linux = 'True' THEN 1 ELSE 0 END) as linux_count,
        COUNT(*) as total
    FROM steam
""", conn)
print("\n" + "=" * 60)
print("QUERY 9: PLATFORM SUPPORT")
print("=" * 60)
print(f"Windows: {q9['windows_count'].values[0]} games")
print(f"Mac:     {q9['mac_count'].values[0]} games")
print(f"Linux:   {q9['linux_count'].values[0]} games")
print(f"Total:   {q9['total'].values[0]} games")

# ============================================================
# QUERY 10: Games with Highest Positive Review %
# ============================================================
q10 = pd.read_sql("""
    SELECT name, positive, negative,
           (positive + negative) as total_reviews,
           ROUND(positive * 100.0 / (positive + negative), 1) as positive_pct,
           price, primary_genre
    FROM steam
    WHERE (positive + negative) > 10000
    ORDER BY positive_pct DESC
    LIMIT 10
""", conn)
print("\n" + "=" * 60)
print("QUERY 10: HIGHEST RATED GAMES (10K+ reviews)")
print("=" * 60)
print(q10.to_string(index=False))

# ============================================================
# SAVE FINAL DATASET
# ============================================================
conn.close()
df_tableau.to_csv('steam_tableau_ready.csv', index=False)

print("\n" + "=" * 60)
print(" ALL 10 SQL QUERIES COMPLETE!")
print("=" * 60)
print(f"Final dataset: {df_tableau.shape[0]} rows x {df_tableau.shape[1]} columns")
print(f"Columns: {df_tableau.columns.tolist()}")
print(f"\nFile saved: steam_tableau_ready.csv")
print("\nNEXT STEP: Open Tableau → Connect to Text File → Select steam_tableau_ready.csv")

QUERY 1: GAMES BY YEAR
 release_year  game_count  avg_price
         2010         167       8.21
         2011         215       8.95
         2012         311      10.19
         2013         464      10.80
         2014        1529       8.85
         2015        2528       7.63
         2016        4163       7.17
         2017        5967       6.78
         2018        7491       6.32
         2019        6159       7.21
         2020        8630       6.58
         2021        8727       7.21
         2022        9926       7.09
         2023       14016       6.76
         2024       20427       6.68
         2025        4121       6.68

QUERY 2: TOP 10 GENRES
primary_genre
['Action'        34269
['Adventure'     17117
['Casual'        15319
['Indie'          5853
[]                5441
['Casual']        2591
['Indie']         2475
['Action']        2283
['Adventure']     1670
['Strategy']      1053

QUERY 3: PRICE DISTRIBUTION
price_category
Free        19420
Under $5    39409


In [9]:
# Fix genre names — remove brackets and quotes
df_tableau['primary_genre'] = df_tableau['primary_genre'].str.replace("[", "", regex=False)
df_tableau['primary_genre'] = df_tableau['primary_genre'].str.replace("]", "", regex=False)
df_tableau['primary_genre'] = df_tableau['primary_genre'].str.replace("'", "", regex=False)
df_tableau['primary_genre'] = df_tableau['primary_genre'].str.strip()

# Also fix publisher and developer
df_tableau['primary_publisher'] = df_tableau['primary_publisher'].str.replace("[", "", regex=False)
df_tableau['primary_publisher'] = df_tableau['primary_publisher'].str.replace("]", "", regex=False)
df_tableau['primary_publisher'] = df_tableau['primary_publisher'].str.replace("'", "", regex=False)
df_tableau['primary_publisher'] = df_tableau['primary_publisher'].str.strip()

df_tableau['primary_developer'] = df_tableau['primary_developer'].str.replace("[", "", regex=False)
df_tableau['primary_developer'] = df_tableau['primary_developer'].str.replace("]", "", regex=False)
df_tableau['primary_developer'] = df_tableau['primary_developer'].str.replace("'", "", regex=False)
df_tableau['primary_developer'] = df_tableau['primary_developer'].str.strip()

# Save updated file
df_tableau.to_csv('steam_tableau_ready.csv', index=False)
print("Fixed! Sample genres:")
print(df_tableau['primary_genre'].value_counts().head(5))
print("\nSample publishers:")
print(df_tableau['primary_publisher'].value_counts().head(5))
print("\nFile saved: steam_tableau_ready.csv")

Fixed! Sample genres:
primary_genre
Action       36552
Adventure    18787
Casual       17910
Indie         8328
              5441
Name: count, dtype: int64

Sample publishers:
primary_publisher
                   5786
Big Fish Games      484
8floor              237
EroticGamesClub     196
Conglomerate 5      187
Name: count, dtype: int64

File saved: steam_tableau_ready.csv
